# Robust Neural Network Learning — 12 April 2026
## Core A* Submission: Loss Functions under Label Noise and Adversarial Perturbations

**Authors**: [add]

This notebook is the interactive companion to `12April2026_RobustNN_Experiments.py`.  
Each section mirrors the theory file `research_writeup/12April2026_RobustNN_Theory_Math.tex`.

---
### Key Research Questions
1. Which loss function is most robust to **label noise** at high η?
2. Which loss function is most robust to **FGSM adversarial attacks** at ε=8/255?
3. Does the **dual robustness** order agree? (Are the same losses good at both?)
4. Does the **S-divergence** (SDIV) unify the other losses? Empirically confirmed?
5. What is the optimal (β, λ) surface for SDIV?

### Main Plot Fix (vs. March 2026 code)
- **Loss curves**: NEVER plot CCE, SCE, SDIV on the same Y-axis — completely different scales  
  - CCE ∈ [0, +∞), initializes at log(C) ≈ 2.3 for C=10  
  - SCE *includes a ×C multiplier* → values ≈ 8–10× larger than CCE for C=10  
  - SDIV *can be negative* depending on (β, λ)  
  - **Fix**: each loss on its own subplot OR normalize per-run to [0,1]  
- **Confusion matrices**: Use `normalize='true'` for recall-per-class view ∈ [0,1]

---
## Section 0 — Environment Setup

In [ ]:
# Install required packages if running in Colab / RunPod
import subprocess, sys

_PKGS = [
    'torch>=2.6',
    'torchvision',
    'transformers>=4.40',
    'datasets',
    'medmnist',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'pandas',
    'tqdm',
    'plotly',      # for interactive 3D surface
    'kaleido',     # for saving plotly figures
]

for pkg in _PKGS:
    try:
        __import__(pkg.split('>=')[0].replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages ready!')

In [ ]:
import os, sys
sys.path.insert(0, '..')

# Add the code directory to path
CODE_DIR = os.path.dirname(os.path.abspath('.'))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Quick mode for notebook (set 0 for full paper run)
os.environ['ROBUST_NN_QUICK_RUN'] = '1'
os.environ['ROBUST_NN_PART']      = 'A'     # Run only Part A (Vision) here
os.environ['ROBUST_NN_VIT_EPOCHS'] = '15'

# Import the full experiment module
import importlib, code.RobustNN_Experiments_12April2026 as exp  # noqa: adjust path as needed
from code.RobustNN_Experiments_12April2026 import (
    CFG, DEVICE, USE_AMP,
    CCELoss, MAELoss, GCELoss, TruncGCELoss, SCELoss,
    DPDLoss, SDIVLoss, TSCCELoss, ForwardCorrectionLoss,
    make_loss_registry, print_loss_scale_table,
    VisionTransformer, build_vit,
    inject_uniform_noise, make_T_uniform, fgsm_attack,
    load_vision_dataset, NumpyImageDataset,
    train_one,
    plot_training_curves, plot_normalized_confusion_matrix,
    plot_robustness_curves, plot_adversarial_curves,
    plot_dual_robustness_frontier, plot_sdiv_surface,
    set_seed,
)
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

# Publication style
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'figure.dpi': 120,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

print(f'Device: {DEVICE}  |  AMP: {USE_AMP}')

---
## Section 1 — What Are Adversarial Attacks?

### 1.1 Intuition

A **adversarial example** is an input $x'$ that is almost identical to $x$ (invisible to a human) but 
causes the classifier $f$ to predict the wrong class.

$$\text{Human sees: same image} \quad \Rightarrow \quad f(x) = \texttt{cat},\ f(x') = \texttt{dog}$$

### 1.2 FGSM (Fast Gradient Sign Method)

The simplest attack: move $x$ in the direction that **maximally increases the loss** in one step.

$$x' = x + \epsilon \cdot \text{sign}\left(\nabla_x \ell(f(x), y)\right)$$

where $\epsilon$ controls the strength. For pixel values in $[0,1]$, typical values: $\epsilon \in \{1/255, 2/255, 4/255, 8/255\}$.

**Key insight**: The SIGN of the gradient (not magnitude) perturbs EVERY input dimension by exactly $\epsilon$. This is deliberately EFFICIENT to attack.

### 1.3 Do Robust Loss Functions Help Against FGSM?

YES — because:
- **CCE** has unbounded gradient: $\nabla_f = -1/p_y$ → explodes as confidence increases → large attack surface
- **GCE, DPD, SDIV** have bounded or dampened gradients: $\nabla_f \sim p_y^{\beta-1}$ → smaller perturbation effect

This is **dual robustness** — the same loss family that helps against label noise also partially helps against adversarial attacks.

In [ ]:
# Visualize: gradient magnitude vs confidence for different losses
import torch, numpy as np, matplotlib.pyplot as plt

py_vals = np.linspace(0.01, 0.999, 300)  # p_y (correct class probability)

grad_cce   = -1.0 / py_vals                    # d(CCE)/dp_y = -1/p_y
grad_gce   = -py_vals**(0.7-1)                 # d(GCE q=0.7)/dp_y = -p_y^(q-1)
grad_mae   = -np.ones_like(py_vals)            # d(MAE)/dp_y = -1 (constant!)
grad_dpd   = -(1+1/0.05) * 0.05 * py_vals**(0.05-1)  # DPD approximation

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(py_vals, np.abs(grad_cce), label='CCE (unbounded!)', color='#1f77b4', linewidth=2)
ax.plot(py_vals, np.abs(grad_gce), label='GCE q=0.7 (bounded)',  color='#2ca02c', linewidth=2)
ax.plot(py_vals, np.abs(grad_mae), label='MAE (constant)',  color='#ff7f0e', linewidth=2, ls='--')
ax.set_xlim(0.01, 1); ax.set_ylim(0, 15)
ax.set_xlabel('Correct-class probability  $p_y$')
ax.set_ylabel(r'$|\partial \ell / \partial p_y|$  (gradient magnitude)')
ax.set_title('Why CCE is Vulnerable to FGSM\n(Gradient explodes as confidence increases)',
             fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
ax.annotate('← explodes here', xy=(0.95, 12), color='#1f77b4', fontsize=10)
plt.tight_layout(); plt.show()

---
## Section 2 — Why Multiple Loss Functions?

### 2.1 The Fundamental Problem: Label Noise

In practice, training labels are **noisy** — human annotators make errors, web-scraped data is imperfect.
Uniform label noise model:
$$\tilde{y} = \begin{cases} y & \text{with probability } 1-\eta \\ \text{uniform}(\{0,\ldots,C-1\}\backslash\{y\}) & \text{with probability } \eta \end{cases}$$

### 2.2 Why CCE Fails Under Noise

**CCE** = $-\log p_y$. As $p_y \to 0$, loss $\to \infty$.  
A noisy label forces the model to push $p_{\tilde{y}} \to 1$ (wrong class), causing **catastrophic overfitting to noise**.

**Gradient**: $\nabla_{p_y} \text{CCE} = -1/p_y$ — UNBOUNDED, grows every epoch.

### 2.3 The Loss Function Zoo and Their Robustness Properties

| Loss | Formula | Gradient | Bounded? | Noise robust? |
|------|---------|----------|----------|--------------|
| CCE | $-\log p_y$ | $-1/p_y$ | No | ✗ (baseline) |
| MAE | $1-p_y$ | $-1$ | Yes [0,1] | ✓✓ (but slow) |
| GCE(q) | $(1-p_y^q)/q$ | $-p_y^{q-1}$ | Yes [0,1/q] | ✓ (tunable) |
| DPD(β) | $\sum_k p_k^{β+1} - \frac{1+β}{β}p_y^β$ | $\sim p_y^{β-1}$ | No (can <0) | ✓✓ |
| SDIV(β,λ) | $\frac{1}{A}\sum p_k^{β+1} - \frac{β+1}{AB}p_y^B$ | $\sim p_y^{B-1}$ | No | ✓✓✓ (unified) |
| SCE | $α\cdot\text{CCE} + β\cdot\text{RCE}$ | mixed | Partial | ✓ |
| TSCCE | Trimmed CCE | same as CCE | No | ✓ (trim noisy) |

**SDIV is the MOST GENERAL**: contains DPD (λ=0), GCE (β→0), CCE (β→0,q→0), MAE (β=1) as special cases.

In [ ]:
# Demonstrate loss values on a simple example
import torch, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt

set_seed(42)
C = 10

# p_y from random to almost-confident
py_vals = np.linspace(0.02, 0.99, 200)

# Loss value vs p_y (other classes share remaining probability uniformly)
def compute_loss_vs_py(loss_module, name):
    vals = []
    for py in py_vals:
        # Softmax-valid distribution: p_y gets `py`, rest split equally
        probs = torch.ones(1, C) * (1 - py) / (C - 1)
        probs[0, 0] = py
        # Recover approximate logits (inverse softmax not unique — use probs directly)
        logits = torch.log(probs.clamp(1e-10))
        target = torch.tensor([0])
        with torch.no_grad():
            try: vals.append(loss_module(logits, target).item())
            except: vals.append(float('nan'))
    return np.array(vals)

losses_to_show = [
    ('CCE',             CCELoss()),
    ('MAE',             MAELoss(C)),
    ('GCE (q=0.7)',     GCELoss(0.7)),
    ('DPD (β=0.05)',    DPDLoss(0.05)),
    ('SDIV (β=0.05,λ=-0.8)', SDIVLoss(0.05, -0.8)),
    ('SCE (α=0.1,β=1)', SCELoss(0.1, 1.0, C)),
]

# *** THE CORRECT WAY: each loss on its own subplot ***
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for (name, loss_fn), ax in zip(losses_to_show, axes.flat):
    vals = compute_loss_vs_py(loss_fn, name)
    ax.plot(py_vals, vals, linewidth=2.5, color='steelblue')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('$p_y$ (correct class probability)', fontsize=9)
    ax.set_ylabel(f'Loss value\n({loss_fn.scale_info[:30]})', fontsize=7.5)
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    ax.grid(alpha=0.3)
    if 'CCE' in name:
        ax.set_ylim(-0.1, 8)
        ax.annotate('UNBOUNDED →', xy=(0.05, 6), fontsize=8, color='red')
    if 'SCE' in name:
        ax.annotate(f'⚠ ×{C} amplified\n  vs CCE!', xy=(0.4, ax.get_ylim()[1]*0.7),
                    fontsize=8, color='red')

fig.suptitle('Loss value vs. correct-class probability $p_y$\n'
             '(Each subplot has its OWN Y-axis — never mix on shared axis!)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('NOTE: SCE has a ×C=10 multiplier making it look ×10 larger than CCE.\n'
      'SDIV can go NEGATIVE for small p_y. This is why separate plots are MANDATORY.')

---
## Section 3 — Do I Need to Train from Scratch?

**Short answer: YES for fair comparison.**

If you reuse weights trained with CCE and then switch to SDIV, you are NOT evaluating SDIV — you are evaluating "CCE with a brief SDIV fine-tuning."  
Each loss function must train from a **random initialization** to measure its true inductive bias.

**Exceptions (intentional)**:
1. **BERT fine-tuning** (Part B): you always start from the pretrained BERT checkpoint. BUT each loss function fine-tunes its OWN copy of BERT from the same checkpoint. This is the standard NLP protocol.
2. **CLIP zero-shot** (Part C): there is no training at all — you compute fixed logits from the frozen CLIP encoder and evaluate different loss functions as METRICS on those logits.

**Why does this answer matter for the paper?**  
Reviewers at Core A* conferences will immediately question experimental validity if losses share model weights.

---
## Section 4 — Full Experiment Runner (Part A: Vision)

This runs the complete battery on MNIST:
- **Battery A**: Clean labels, all loss functions, training curves  
- **Battery B**: Uniform label noise η ∈ {0.1, 0.2, 0.3, 0.4}  
- **Battery C**: FGSM adversarial evaluation  
- **Battery E**: SDIV (β, λ) accuracy surface

All plots saved to `results_12April2026/`.

In [ ]:
# Load dataset (MNIST by default)
X_tr, y_tr, X_te, y_te = load_vision_dataset('mnist')
num_classes = 10; C_ch = 1; img_size = 32

print(f'Train: {X_tr.shape}  Test: {X_te.shape}  Classes: {num_classes}')

In [ ]:
# Battery A: Train all losses on CLEAN MNIST labels
registry = make_loss_registry(num_classes, None, 0.7, 0.05, -0.8)
print_loss_scale_table(num_classes, registry)

clean_results = []
for loss_name, loss_fn in registry.items():
    model = build_vit(img_size, C_ch, num_classes)
    res = train_one(
        model, loss_fn,
        X_tr, y_tr, X_te, y_te,
        n_epochs=CFG['VIT_EPOCHS'],
        batch_size=CFG['VIT_BATCH'],
        lr=CFG['VIT_LR'],
        loss_name=loss_name,
        dataset_name='mnist',
        noise_rate=0.0,
        seed=42,
        save_model=True,
    )
    clean_results.append(res)

In [ ]:
# Plot training curves — CORRECT way (each loss in its own subplot)
plot_training_curves(
    clean_results,
    title_suffix='MNIST | Clean Labels',
    save_path='results_12April2026/notebook_mnist_clean_training.png',
)

# Show in notebook
from IPython.display import Image
Image('results_12April2026/notebook_mnist_clean_training.png')

In [ ]:
# Summary table: clean performance
summary = pd.DataFrame([
    {'Loss': r.loss_name, 'Best_Acc': r.best_acc,
     'Final_TrainLoss': r.history['train_loss'][-1],
     'Scale_Info': registry[r.loss_name].scale_info[:40] if r.loss_name in registry else ''}
    for r in clean_results
]).sort_values('Best_Acc', ascending=False)
print('\nClean MNIST — Summary:')
print(summary.to_string(index=False))
summary.to_csv('results_12April2026/notebook_mnist_clean_summary.csv', index=False)

In [ ]:
# Normalized confusion matrices for top-3 losses
import os
os.makedirs('results_12April2026', exist_ok=True)

class_names = [str(i) for i in range(10)]
top3 = summary.head(3)['Loss'].tolist()

for res in clean_results:
    if res.loss_name in top3 and res.best_preds:
        plot_normalized_confusion_matrix(
            res.true_labels, res.best_preds,
            class_names,
            title=f'{res.loss_name} — MNIST Clean',
            save_path=f'results_12April2026/notebook_confmat_{res.loss_name.replace("/","_")}.png'
        )
        Image(f'results_12April2026/notebook_confmat_{res.loss_name.replace("/","_")}.png')

In [ ]:
# Battery B: Noisy label training at η=0.2 and η=0.4
noise_rows = []
for eta in [0.0, 0.2, 0.4]:
    y_noisy = inject_uniform_noise(y_tr, eta, num_classes, seed=42)
    T_oracle = make_T_uniform(num_classes, eta) if eta > 0 else None
    reg_run = make_loss_registry(num_classes, T_oracle, 0.7, 0.05, -0.8)

    for loss_name, loss_fn in reg_run.items():
        model = build_vit(img_size, C_ch, num_classes)
        res = train_one(
            model, loss_fn, X_tr, y_noisy, X_te, y_te,
            n_epochs=CFG['VIT_EPOCHS'], batch_size=CFG['VIT_BATCH'],
            lr=CFG['VIT_LR'],
            loss_name=loss_name, dataset_name='mnist',
            noise_rate=eta, seed=42,
        )
        noise_rows.append({'loss': loss_name, 'noise_rate': eta, 'accuracy': res.best_acc})

noise_df = pd.DataFrame(noise_rows)
print('\nNoise robustness summary:')
pivot = noise_df.pivot(index='loss', columns='noise_rate', values='accuracy').round(4)
pivot.columns = [f'η={c}' for c in pivot.columns]
print(pivot.to_string())

In [ ]:
# Robustness curve plot
plot_robustness_curves(
    noise_df, 'MNIST',
    'results_12April2026/notebook_mnist_noise_robustness.png'
)
Image('results_12April2026/notebook_mnist_noise_robustness.png')

---
## Section 5 — SDIV Unification: Analytical Demo

**Theorem (S-divergence superfamily)**:  
Let $A = 1 + \lambda(1-\beta)$, $B = \beta - \lambda(1-\beta)$. Then:
- $\text{SDIV}_{\beta, \lambda=0} = \text{DPD}(\beta)$  
- $\lim_{\beta \to 0} \text{SDIV}_{\beta, \lambda} = \text{GCE}(q)$ where $q = B$  
- $\lim_{\beta \to 0, q \to 0} \text{GCE}(q) = \text{CCE}$  
- $\text{GCE}(q=1) = \text{MAE}$

So the lattice of inclusions is: $\text{SDIV} \supset \text{DPD} \supset \text{GCE} \supset \{\text{CCE}, \text{MAE}\}$

In [ ]:
# Show numerical convergence: SDIV(β→0, λ=0) ≈ GCE(q=β) for small β
import torch, numpy as np

set_seed(42)
torch.manual_seed(42)
C = 10; B = 32
logits = torch.randn(B, C)
targets = torch.randint(0, C, (B,))

print('Numerical verification: SDIV(β, λ=0) ≈ DPD(β) (should match):')
for beta in [0.02, 0.05, 0.10, 0.20, 0.50]:
    dpd_val  = DPDLoss(beta)(logits, targets).item()
    sdiv_val = SDIVLoss(beta, lam=0.0)(logits, targets).item()
    diff = abs(dpd_val - sdiv_val)
    print(f'  β={beta:.2f} | DPD={dpd_val:+.6f} | SDIV(λ=0)={sdiv_val:+.6f} | diff={diff:.2e}  {"✓" if diff < 1e-4 else "✗"}')

print('\nGCE(q=1) vs MAE (should match):')
gce1  = GCELoss(q=1.0)(logits, targets).item()
mae   = MAELoss(C)(logits, targets).item()
print(f'  GCE(q=1)={gce1:.6f} | MAE={mae:.6f} | diff={abs(gce1-mae):.2e}')

---
## Section 6 — Dual Robustness: Label Noise + FGSM Together

The central novel claim: a loss that is robust to label noise $\eta$ is ALSO more robust to FGSM $\epsilon$, because they share the same mechanism — **gradient bounding**.

The **dual robustness frontier** scatter plot shows this — losses in the top-right corner are Pareto-optimal (best on both axes).

In [ ]:
# Battery C: FGSM evaluation on models trained with clean labels
from torch.utils.data import DataLoader

fgsm_rows = []
cce_loss = CCELoss()

for res in clean_results:
    if res.model_state is None: continue
    model = build_vit(img_size, C_ch, num_classes)
    model.load_state_dict(res.model_state)
    model.eval()

    te_loader = DataLoader(NumpyImageDataset(X_te, y_te), 256, shuffle=False)
    for eps in [0.0, 2/255, 4/255, 8/255]:
        preds, labs = [], []
        for xb, yb in te_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            x_adv = fgsm_attack(model, xb, yb, eps, cce_loss)
            with torch.no_grad():
                out = model(x_adv)
            preds.extend(out.argmax(1).cpu().tolist())
            labs.extend(yb.cpu().tolist())
        from sklearn.metrics import accuracy_score
        acc = accuracy_score(labs, preds)
        fgsm_rows.append({'loss': res.loss_name, 'epsilon': eps, 'accuracy': acc})

fgsm_df = pd.DataFrame(fgsm_rows)
print('\nFGSM robustness summary (accuracy at ε=8/255):')
print(fgsm_df[fgsm_df['epsilon']==8/255][['loss','accuracy']].sort_values('accuracy', ascending=False).to_string(index=False))

In [ ]:
# Dual robustness frontier
plot_dual_robustness_frontier(
    noise_df, fgsm_df,
    dataset='MNIST',
    noise_rate=0.2, epsilon=4/255,
    save_path='results_12April2026/notebook_mnist_dual_frontier.png',
)
Image('results_12April2026/notebook_mnist_dual_frontier.png')

---
## Section 7 — Interactive SDIV (β, λ) Accuracy Surface (Plotly)

The key insight: not all (β, λ) combinations are valid. We need A = 1+λ(1-β) > 0 and B = β-λ(1-β) > 0.  
The accuracy surface over this feasible region tells us the optimal hyperparameters.

In [ ]:
# Quick SDIV surface (fewer points for notebook speed)
import pandas as pd
import importlib.util

beta_grid = [0.02, 0.05, 0.10, 0.20]
lam_grid  = [-0.8, -0.4, 0.0]

sdiv_rows = []
for beta in beta_grid:
    for lam in lam_grid:
        A = 1.0 + lam*(1-beta); B = beta - lam*(1-beta)
        if A <= 0 or B <= 0: continue
        try:
            loss_fn = SDIVLoss(beta, lam)
        except ValueError: continue
        m = build_vit(img_size, C_ch, num_classes)
        res = train_one(
            m, loss_fn, X_tr, y_tr, X_te, y_te,
            n_epochs=CFG['VIT_EPOCHS'], batch_size=CFG['VIT_BATCH'],
            lr=CFG['VIT_LR'],
            loss_name=f'SDIV(β={beta},λ={lam})',
            dataset_name='mnist', noise_rate=0.0, seed=42,
        )
        sdiv_rows.append({'beta': beta, 'lam': lam, 'accuracy': res.best_acc})

sdiv_df = pd.DataFrame(sdiv_rows)
print(sdiv_df.sort_values('accuracy', ascending=False).to_string(index=False))

In [ ]:
# Interactive 3D surface with Plotly
try:
    import plotly.graph_objects as go
    pivot = sdiv_df.pivot(index='beta', columns='lam', values='accuracy')
    fig = go.Figure(data=[go.Surface(
        x=pivot.columns.values, y=pivot.index.values, z=pivot.values,
        colorscale='Viridis', showscale=True,
    )])
    fig.update_layout(
        title='SDIV Accuracy Surface — MNIST Clean Labels',
        scene=dict(xaxis_title='λ (lam)', yaxis_title='β (beta)', zaxis_title='Accuracy'),
        width=750, height=600,
    )
    fig.write_html('results_12April2026/sdiv_surface_interactive.html')
    fig.show()
    print('Interactive surface saved to results_12April2026/sdiv_surface_interactive.html')
except ImportError:
    print('Plotly not installed — using matplotlib fallback')
    plot_sdiv_surface(sdiv_df, 'MNIST', 0.0, 'results_12April2026/sdiv_surface_static.png')
    Image('results_12April2026/sdiv_surface_static.png')

---
## Section 8 — Curriculum GCE Annealing (Novel Contribution)

**Idea**: GCE(q) interpolates between robust (high q) and efficient (low q).  
Instead of fixing q, **anneal it** over training:
$$q(t) = q_{\max} \cdot \left(1 - \frac{t}{T}\right)$$

- Early epochs: high q → MAE-like → robust against early noisy memorization  
- Late epochs: low q → CCE-like → precise fine-tuning on clean signal

This is a **curriculum learning** strategy inspired by self-paced learning.

In [ ]:
from code.RobustNN_Experiments_12April2026 import run_curriculum_annealing
run_curriculum_annealing('mnist', eta=0.3, seed=42)
Image('results_12April2026/mnist_curriculum_gce_eta0.3.png')

---
## Section 9 — Results Summary Table (Paper-Ready)

Reconstruct the final table to put in the .tex file.

In [ ]:
# Build final comparison table
table_rows = []
for res in clean_results:
    clean_acc = res.best_acc
    noisy_acc_02 = noise_df[(noise_df['loss']==res.loss_name) & (noise_df['noise_rate']==0.2)]['accuracy'].values
    noisy_acc_04 = noise_df[(noise_df['loss']==res.loss_name) & (noise_df['noise_rate']==0.4)]['accuracy'].values
    fgsm_acc    = fgsm_df[(fgsm_df['loss']==res.loss_name) & (fgsm_df['epsilon']==8/255)]['accuracy'].values
    table_rows.append({
        'Loss': res.loss_name,
        'Clean Acc': f'{clean_acc:.4f}',
        'Noise η=0.2': f'{noisy_acc_02[0]:.4f}' if len(noisy_acc_02)>0 else 'N/A',
        'Noise η=0.4': f'{noisy_acc_04[0]:.4f}' if len(noisy_acc_04)>0 else 'N/A',
        'FGSM ε=8/255': f'{fgsm_acc[0]:.4f}' if len(fgsm_acc)>0 else 'N/A',
    })

final_table = pd.DataFrame(table_rows)
print('\n📊 MNIST — Final Results Table (12 April 2026)')
print('='*80)
print(final_table.to_string(index=False))
final_table.to_csv('results_12April2026/MNIST_final_table_12April2026.csv', index=False)
print('\nSaved to results_12April2026/MNIST_final_table_12April2026.csv')

---
## Section 10 — Generate LaTeX Table from Results

In [ ]:
# Auto-generate LaTeX table snippet
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{MNIST Robustness Benchmark (12 April 2026). '
      r'Best values \textbf{bold}, second \underline{underlined}.}')
print(r'\label{tab:mnist-results}')
print(r'\begin{tabular}{lcccc}')
print(r'\toprule')
print(r'Loss & Clean & $\eta=0.2$ & $\eta=0.4$ & FGSM $\epsilon=8/255$ \\')
print(r'\midrule')
for _, row in final_table.iterrows():
    print(f"{row['Loss']} & {row['Clean Acc']} & {row['Noise η=0.2']} "
          f"& {row['Noise η=0.4']} & {row['FGSM ε=8/255']} \\\\")
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

---
## Section 11 — Next Steps for Core A* Submission

Based on current results, the paper's novel contributions to highlight:

1. **Unification theorem**: SDIV(β,λ) contains DPD, GCE, CCE, MAE as special cases — formally proved in theory doc
2. **Dual robustness theorem**: Joint bound on accuracy under (η noise rate, ε FGSM) — novel Theorem 7.1
3. **Dual robustness frontier visualization**: First paper to show scatter of (noise acc, FGSM acc) per loss
4. **Curriculum GCE annealing**: Novel q(t) schedule that outperforms fixed q in noisy settings
5. **Cross-modal evaluation**: Same loss function family evaluated on ViT (vision), BERT (NLP), CLIP (zero-shot multimodal)

**Key message for reviewers**: S-divergence provides a principled, unified framework for robust deep learning — unlike ad-hoc loss collections in prior work.

**Suggested venue**: ICML 2026 (deadline: January 2026), NeurIPS 2026 (May), ICLR 2027 (October)